In [0]:
# Testing table joins
df = spark.sql("""
    SELECT
        d.date_id,
        h.hour_id,
        c.company_id,
        zpu.zone_id AS pu_zone_id,
        zdo.zone_id AS do_zone_id,
        COUNT(*) AS total_trips,
        SUM(t.trip_distance) AS total_trip_distance,
        SUM(t.trip_time) AS total_trip_time
    FROM group3_gp.silver.combined_taxi_trips t
    INNER JOIN group3_gp.gold.dim_date d 
        ON CAST(t.pickup_datetime AS DATE) = d.date
    JOIN group3_gp.gold.dim_company c
        ON t.service_type = c.service_type
        AND t.taxi_type = c.taxi_type
    JOIN group3_gp.gold.dim_hour h
        ON hour(t.pickup_datetime) = h.hour_id
    JOIN group3_gp.gold.dim_zone zpu
        ON t.pulocationid = zpu.zone_id
    JOIN group3_gp.gold.dim_zone_dropoff zdo
        ON t.dolocationid = zdo.zone_id
    GROUP BY d.date_id, c.company_id, h.hour_id, zpu.zone_id, zdo.zone_id
    ORDER BY d.date_id
""")

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("group3_gp.testing.Fact_Grouped_By_Hour_Date_Company_Zone")

Below are the fact tables being created in the notebook and their locations:

- `group3_gp.testing.Fact_Grouped_By_Hour_Date_Company_Zone` (testing schema)
- `group3_gp.gold.1_Fact_Revenue_And_Fares` (gold schema)
- `group3_gp.gold.2_Fact_Trip_Demand_Volume` (gold schema)
- `group3_gp.gold.4_Fact_Company_and_Market_Share` (gold schema)
- `group3_gp.gold.5_Fact_Trip_Efficiency_and_Operations` (gold schema)
- `group3_gp.gold.3_and_7_Fact_Geography_and_Zones_and_Covid` (gold schema)

In [0]:
# 1. Fact Revenue and Fares
df = spark.sql("""
    SELECT
        d.date_id,
        c.company_id,
        t.payment_types,
        COUNT(*) AS total_trips,
        SUM(t.trip_distance) AS total_trip_distance,
        AVG(t.trip_distance) AS avg_trip_distance,
        SUM(t.trip_time) AS total_trip_time,
        AVG(t.trip_time) AS avg_trip_time,
        SUM(t.total_amount) AS total_revenue,
        AVG(t.total_amount) AS avg_revenue,
        SUM(t.tip_amount) AS total_tip,
        avg(t.tip_amount) AS avg_tip
    FROM group3_gp.silver.combined_taxi_trips t
    JOIN group3_gp.gold.dim_date d
        ON date(t.pickup_datetime) = d.date
    JOIN group3_gp.gold.dim_company c
        ON t.service_type = c.service_type
        AND t.taxi_type = c.taxi_type
    GROUP BY d.date_id, c.company_id, t.payment_types
    ORDER BY d.date_id
""")

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("group3_gp.gold.1_Fact_Revenue_And_Fares")

In [0]:
# 2. Trip Demand and Volume
df = spark.sql("""
    SELECT
        d.date_id,
        h.hour_id,
        c.company_id,
        COUNT(*) AS total_trips,
        SUM(t.trip_distance) AS total_trip_distance,
        AVG(t.trip_distance) AS avg_trip_distance,
        SUM(t.trip_time) AS total_trip_time,
        AVG(t.trip_time) AS avg_trip_time
    FROM group3_gp.silver.combined_taxi_trips t
    JOIN group3_gp.gold.dim_date d
        ON date(t.pickup_datetime) = d.date
    JOIN group3_gp.gold.dim_company c
        ON t.service_type = c.service_type
        AND t.taxi_type = c.taxi_type
    JOIN group3_gp.gold.dim_hour h
        ON hour(t.pickup_datetime) = h.hour_id
    GROUP BY d.date_id, c.company_id, h.hour_id
    ORDER BY d.date_id
""")

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("group3_gp.gold.2_Fact_Trip_Demand_Volume")


In [0]:
# 4. Company and market share
df = spark.sql("""
    SELECT
        d.date_id,
        c.company_id,
        COUNT(*) AS total_trips,
        SUM(t.total_amount) AS total_revenue,
        AVG(t.total_amount) AS avg_revenue,
        SUM(t.tip_amount) AS total_tip,
        AVG(t.tip_amount) AS avg_tip,
        SUM(t.trip_distance) AS total_trip_distance,
        AVG(t.trip_distance) AS avg_trip_distance,
        SUM(t.trip_time) AS total_trip_time,
        AVG(t.trip_time) AS avg_trip_time
    FROM group3_gp.silver.combined_taxi_trips t
    JOIN group3_gp.gold.dim_date d
        ON date(t.pickup_datetime) = d.date
    JOIN group3_gp.gold.dim_company c
        ON t.service_type = c.service_type
        AND t.taxi_type = c.taxi_type
    GROUP BY d.date_id, c.company_id
    ORDER BY d.date_id
""")

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("group3_gp.gold.4_Fact_Company_and_Market_Share")

In [0]:
# 5. Trip Efficiency and Operations
df = spark.sql("""
    SELECT
        c.company_id,
        zpu.zone_id AS pu_zone_id,
        zdo.zone_id AS do_zone_id,
        COUNT(*) AS total_trips,
        SUM(t.trip_distance) AS total_trip_distance,
        AVG(t.trip_distance) AS avg_trip_distance,
        SUM(t.trip_time) AS total_trip_time,
        AVG(t.trip_time) AS avg_trip_time,
        SUM(t.total_amount) AS total_amount,
        AVG(t.total_amount) AS avg_amount
    FROM group3_gp.silver.combined_taxi_trips t
    JOIN group3_gp.gold.dim_company c
        ON t.service_type = c.service_type
        AND t.taxi_type = c.taxi_type
    JOIN group3_gp.gold.dim_zone zpu
        ON t.pulocationid = zpu.zone_id
    JOIN group3_gp.gold.dim_zone_dropoff zdo
        ON t.dolocationid = zdo.zone_id
    GROUP BY c.company_id, zpu.zone_id, zdo.zone_id
    ORDER BY c.company_id
""")

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("group3_gp.gold.5_Fact_Trip_Efficiency_and_Operations")

In [0]:
# 3. Geography and Zones
# 7. Covid impact
df = spark.sql("""
    SELECT
        d.Year,
        c.company_id,
        zpu.zone_id AS pu_zone_id,
        zdo.zone_id AS do_zone_id,
        d.is_covid,
        COUNT(*) AS total_trips,
        SUM(t.trip_distance) AS total_trip_distance,
        AVG(t.trip_distance) AS avg_trip_distance,
        SUM(t.trip_time) AS total_trip_time,
        AVG(t.trip_time) AS avg_trip_time,
        SUM(t.total_amount) AS total_amount,
        AVG(t.total_amount) AS avg_amount
    FROM group3_gp.silver.combined_taxi_trips t
    JOIN group3_gp.gold.dim_date d
        ON date(t.pickup_datetime) = d.date
    JOIN group3_gp.gold.dim_company c
        ON t.service_type = c.service_type
        AND t.taxi_type = c.taxi_type
    JOIN group3_gp.gold.dim_zone zpu
        ON t.pulocationid = zpu.zone_id
    JOIN group3_gp.gold.dim_zone_dropoff zdo
        ON t.dolocationid = zdo.zone_id
    GROUP BY d.Year, c.company_id, zpu.zone_id, zdo.zone_id, d.is_covid
    ORDER BY d.Year
""")

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("group3_gp.testing.3_and_7_Fact_Geography_and_Zones_and_Covid")